- 둘이 정밀도 차이가 좀 나네??? 왜그러지???

In [24]:
import sys, os
from collections import OrderedDict

import numpy as np
sys.path.append('./official_github/')
from dataset.mnist import load_mnist
from my_functions import *
from my_layers import *

In [25]:
class TwoLayerNet():
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01) -> None:
        
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)
        
    
    def predict(self, x):
        a1 = sigmoid(x @ self.params['W1'] + self.params['b1'])
        a2 = sigmoid(a1 @ self.params['W2'] + self.params['b2'])
        out = softmax(a2)
        return out
    
    def loss(self, x, t):
        y = self.predict(x)
        return cross_entropy_error(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        accuracy = np.sum(y == t) / x.shape[0]
        
    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)
        
        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])
        
        return grads
    

In [26]:
d = {'a':1, 'b':2, 'c':3}
d = list(d.values())
d.reverse()
print(d)


d = {'a':1, 'b':2, 'c':3}
list(reversed(d.values()))

[3, 2, 1]


[3, 2, 1]

In [27]:
class TwoLayerNet():
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01) -> None:
        
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)
        
        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = ReLU()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])
        # self.layers['Relu2'] = ReLU()
        
        self.last_layer = SoftmaxWithLoss()
    
        
    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)
        return x
    
    
    def loss(self, x, t):
        y = self.predict(x)

        loss = self.last_layer.forward(y, t)
        return loss
    
    
    def accuracy(self, x, t):
        y = self.predict(x)
        if t.ndim != 1:     ## 원-핫 라벨인 경우, 다시 1열짜리 매트릭스로 전환.
            t = np.argmax(t, axis=1)
        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy
    
    
    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)
        
        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])
        
        return grads
    
    
    def gradient(self, x, t):
        
        self.loss(x, t)
        
        dout = 1
        dout = self.last_layer.backward(dout)
        reversed_layers = reversed(self.layers.values())
        for layer in reversed_layers:
            print(layer)
            dout = layer.backward(dout)
            
        grads = {}
        grads['W1'] = self.layers['Affine1'].dW
        grads['b1'] = self.layers['Affine1'].db
        grads['W2'] = self.layers['Affine2'].dW
        grads['b2'] = self.layers['Affine2'].db
        
        return grads

In [28]:
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

epoch = 1
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

network = TwoLayerNet(784, 50, 10)

In [29]:
x_batch, t_batch = x_train[:3], t_train[:3]

grad_numerical = network.numerical_gradient(x_batch, t_batch)
grad_backprop = network.gradient(x_batch, t_batch)

for key in grad_numerical.keys():
    diff = np.average(np.abs(grad_numerical[key] - grad_backprop[key]))
    print(f'{key} diff is {diff}' )

W1 diff is 0.0013274918412811262
b1 diff is 0.009209926188368127
W2 diff is 0.015089241110615924
b2 diff is 0.38699331694065375


In [30]:
# def data_loader(data: np.ndarray, batch_size=100) -> np.ndarray:
#     batch_indexes= np.arange(len(data))
#     # np.random.shuffle(batch_indexes)
#     while batch_indexes.any():
#         batch = batch_indexes[:batch_size]
#         # print(batch)
#         batch_indexes = batch_indexes[batch_size:]
#         yield data[batch, ...]


# train_loss_list = []
# for epoch_idx in range(1, epoch+1):
#     print(f"===== {epoch_idx} epoch started =====")
#     x_train_loader = data_loader(x_train)
#     t_train_loader = data_loader(t_train)
    
#     for x_batch, t_batch in zip(x_train_loader, t_train_loader):
#         print(x_batch.shape, t_batch.shape)
#         print('predict x batch shape', network.predict(x_batch).shape)
#         grad = network.numerical_gradient(x_batch, t_batch)
        
#         for key in network.params.keys():
#             network.params[key] -= learning_rate * grad[key]
    
#     loss = network.loss(x_batch, t_batch)
#     print('loss:', loss)
#     train_loss_list.append(loss)
    
    